<a href="https://colab.research.google.com/github/pcmay/ALyzer3D.AI/blob/main/ALyzer3DAI_batch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>




<div style="display: flex; justify-content: space-between; align-items: center;">
<img src="https://raw.githubusercontent.com/petercmay89/ALyzer3D.AI/main/white.png" width="10%">
<img src="https://raw.githubusercontent.com/petercmay89/ALyzer3D.AI/main/ALyzer3D.AI_logo.png" width="25%">
<img src="https://raw.githubusercontent.com/petercmay89/ALyzer3D.AI/main/white.png" width="25%">
<img src="https://raw.githubusercontent.com/petercmay89/ALyzer3D.AI/main/ColabFold_logo.png" width="25%">
<img src="https://raw.githubusercontent.com/petercmay89/ALyzer3D.AI/main/white.png" width="10%">
</div>



Welcome to **ALyzer3D.AI BATCH**. This notebook allows you to predict the amyloidogenicity of the VL domains of a list of light chains. The tool will first generate 3D structures with [ColabFold](https://colab.research.google.com/github/sokrypton/ColabFold/blob/main/AlphaFold2.ipynb) and then automatically analyze them with the ALyzer3D.AI model.

**Instructions:**

1. **Enter Your Sequences**: In the first cell (sequences_input), paste the amino acid sequences of your light chains' VL domains. Format: >ID1:sequence1;>ID2:sequence2;>ID3:sequence3;...
2. **Select a GPU**: Click Runtime, select Change runtime type, select T4 GPU (or any GPU option available - DO NOT use TPUs). Click Save.
3. **Run Everything**: Click on the menu Runtime -> Run all.

The notebook will now execute all the steps for you: it will install dependencies, run the ColabFold structure predictions and perform the ALyzer3D.AI analyses on the resulting top-ranked structures. At Step 3, you will be able to download a CSV file with the results.


---



In [ ]:
#@title Install Dependencies
import os, sys, subprocess

if not os.path.isfile("COLABFOLD_READY"):
    print("Installing ColabFold...")
    os.system("pip install -q --no-warn-conflicts "
              "'colabfold[alphafold-minus-jax] @ git+https://github.com/sokrypton/ColabFold'")
    # NOTE: the old 'rm -f .../libtfkernel_sobol_op.so' line is gone on purpose.
    # Deleting a library out of dist-packages to work around an import error
    # leaves a damaged TensorFlow behind and causes silent segfaults later.
    os.system("ln -s /usr/local/lib/python3.*/dist-packages/colabfold colabfold")
    os.system("ln -s /usr/local/lib/python3.*/dist-packages/alphafold alphafold")
    os.system("touch COLABFOLD_READY")

print("Installing ALyzer3D.AI and its dependencies...")
if not os.path.exists("/content/ALyzer3D.AI"):
    subprocess.run(["git", "clone", "https://github.com/petercmay89/ALyzer3D.AI.git",
                    "/content/ALyzer3D.AI"], check=True, capture_output=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "transformers", "scikit-learn", "joblib", "biopython", "pandas"],
               check=True, capture_output=True)
print("Done.")

In [ ]:
#@title Run Batch Prediction and Analysis

#@markdown ### Paste your sequences below (format: `>ID1:SEQUENCE;>ID2:SEQUENCE`)
sequences_input = '>ID1:DIRLTQSPSSLSASVGDRVTITCQASQHINNYLNWYQHKPGQAPKVLIYDASNLATGVPSRFSGNGSGTHFTLTINSLQPEDAATYYCQQHDDLPLTFGGGTKVEIR;>ID2:SASASLGASVNFTCTLSNEHSTYAITWHQQQPKKGPRYLMKVKSDGSHNKGDGIPDRFSGSSSGAERYLTISSLQSDNEADYYCQTWDTDILVFGGGTNLTVL' #@param {type:"string"}
VERBOSE = False

output_dir = 'colabfold_output'

# Decision threshold is fixed at 0.60 (Youden-optimised for seed 3, paper Table 2)
# and defined once inside the worker. It is deliberately not user-tunable.

# --- Standard Parameters ---
num_relax = 0
template_mode = "none"
msa_mode = "mmseqs2_uniref_env"
model_type = "auto"
pair_mode = "unpaired_paired"
num_recycles = 3
# ---------------------------

# TensorFlow/Keras are deliberately NOT imported here: this kernel hosts
# JAX/ColabFold and owns the CUDA context. Analysis runs in a subprocess.
import os, re, sys, glob, json, subprocess, warnings
from pathlib import Path

warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=UserWarning)

from colabfold.download import download_alphafold_params
from colabfold.utils import setup_logging
from colabfold.batch import get_queries, run, set_model_type

REPO_PATH = "/content/ALyzer3D.AI"
MODEL_FOLDER_NAME = "paper_model_scalar_pathway_v1_minus5_stripped_80_20_seed3"
SCRIPT_PATH = "/content/run_alyzer_batch.py"
MANIFEST_PATH = "/content/_alyzer_manifest.json"
RESULTS_PATH = "/content/_alyzer_batch_raw.json"

def log(msg):
    if VERBOSE:
        print(msg)

# ------------------------------------------------------------------------------
# 1. PARSE INPUT
# ------------------------------------------------------------------------------
sequences_to_process = []
seen_ids = set()
processed_input = sequences_input.strip().lstrip('>')

for entry in processed_input.split(';'):
    entry = entry.strip()
    if not entry:
        continue
    if ':' not in entry:
        print(f"⚠️ Skipping malformed entry (missing ':'): '{entry[:40]}'")
        continue
    qid, seq = (p.strip() for p in entry.split(':', 1))
    qid = qid.lstrip('>')
    seq = "".join(seq.split()).upper()
    if not qid or not seq:
        print(f"⚠️ Skipping malformed entry: '{entry[:40]}'")
        continue
    bad = set(seq) - set("ACDEFGHIKLMNPQRSTVWY")
    if bad:
        print(f"⚠️ '{qid}' contains non-standard residues {sorted(bad)} — skipped.")
        continue
    # Distinct IDs can collapse to the same folder name after sanitising.
    safe = re.sub(r'\W+', '', qid)
    if safe in seen_ids:
        print(f"⚠️ '{qid}' collides with an earlier ID as '{safe}' — skipped.")
        continue
    seen_ids.add(safe)
    sequences_to_process.append({'id': qid, 'safe': safe, 'sequence': seq})

if not sequences_to_process:
    raise ValueError("No valid sequences parsed from input.")
print(f"✅ Found {len(sequences_to_process)} sequences to process.")

# ------------------------------------------------------------------------------
# 2. FOLD EVERYTHING (in-kernel, GPU)
# ------------------------------------------------------------------------------
os.makedirs(output_dir, exist_ok=True)
all_results = []   # consumed by cell 3
manifest = []

_params_ready = False
for item in sequences_to_process:
    jobname = os.path.join(output_dir, item['safe'])
    os.makedirs(jobname, exist_ok=True)
    queries_path = os.path.join(jobname, f"{item['safe']}.csv")
    with open(queries_path, "w") as f:
        f.write(f"id,sequence\n{item['safe']},{item['sequence']}")

    print(f"\n{'='*60}\nFolding: {item['id']} ({len(item['sequence'])} aa)")

    result_dir = Path(jobname)
    setup_logging(result_dir.joinpath("log.txt"))
    queries, is_complex = get_queries(queries_path)
    model_type_run = set_model_type(is_complex, model_type)

    if not _params_ready:          # cached, but no reason to re-check per sequence
        download_alphafold_params(model_type_run, Path("."))
        _params_ready = True

    try:
        run(
            queries=queries, result_dir=result_dir,
            use_templates=(template_mode != "none"), custom_template_path=None,
            num_relax=num_relax, msa_mode=msa_mode, model_type=model_type_run,
            num_models=5, num_recycles=int(num_recycles), num_seeds=1,
            model_order=[1, 2, 3, 4, 5], is_complex=is_complex,
            data_dir=Path("."), keep_existing_results=False, rank_by="auto",
            pair_mode=pair_mode, stop_at_score=100.0, zip_results=False,
            user_agent="colabfold/google-colab-main",
        )
    except Exception as exc:
        print(f"❗️ ColabFold failed for {item['id']}: {exc}")
        all_results.append({"ID": item['id'], "Sequence": item['sequence'],
                            "Prediction": "Processing Error", "Probability": 0.0,
                            "Notes": f"ColabFold error: {exc}"})
        continue

    pdb_file = next(Path(jobname).glob("*_unrelaxed_rank_001*.pdb"), None)
    json_file = next(Path(jobname).glob("*_scores_rank_001*.json"), None)
    if not (pdb_file and json_file):
        print(f"❗️ No rank_001 output for {item['id']}.")
        all_results.append({"ID": item['id'], "Sequence": item['sequence'],
                            "Prediction": "Processing Error", "Probability": 0.0,
                            "Notes": "ColabFold failed to generate output"})
        continue

    manifest.append({"id": item['id'], "input_sequence": item['sequence'],
                     "pdb": str(pdb_file), "json": str(json_file)})

if not manifest:
    raise RuntimeError("No structures were produced — nothing to analyse.")

# ------------------------------------------------------------------------------
# 3. WORKER SCRIPT
# ------------------------------------------------------------------------------
WORKER = r'''
import os, sys, re, glob, json
import numpy as np, joblib, torch, keras

MODEL_DIR, MANIFEST, OUT_PATH, VERBOSE = sys.argv[1:5]
VERBOSE = VERBOSE == "1"
MAX_LENGTH = 120
THRESHOLD = 0.60          # Youden-optimised for seed 3 (paper, Table 2)
PLM_MODEL_NAME = "facebook/esm2_t6_8M_UR50D"

def log(msg):
    if VERBOSE:
        print(f"[worker] {msg}", flush=True)

if not keras.__version__.startswith("3"):
    sys.exit(f"Keras 3 required, found {keras.__version__}")
log(f"keras {keras.__version__}")

from transformers import AutoTokenizer, EsmModel
from Bio.PDB import PDBParser
from Bio.PDB.Polypeptide import is_aa
from Bio.Data.PDBData import protein_letters_3to1
from Bio.SeqUtils.ProtParam import ProteinAnalysis

# --- Lambda compat: 'normalize_length' is stored as marshalled bytecode from a
# --- different Python version. marshal.loads() on that can segfault the
# --- interpreter outright, so intercept BEFORE deserialization is attempted.
# --- A try/except cannot help here: a segfault is not an exception.
_ORIG = keras.layers.Lambda.from_config.__func__
_BYPASSED = []

def _safe(cls, config, *a, **kw):
    fn = config.get("function")
    if isinstance(fn, dict) and fn.get("class_name") == "__lambda__":
        closure = fn.get("config", {}).get("closure") or []
        divisor = float(closure[0]) if closure else float(MAX_LENGTH)
        _BYPASSED.append(divisor)
        log(f"bypassing marshalled Lambda '{config.get('name')}' (x / {divisor})")

        def _normalize_length(x, _d=divisor):
            return x / _d

        return cls(function=_normalize_length, name=config.get("name"),
                   trainable=config.get("trainable", True))
    return _ORIG(cls, config, *a, **kw)

keras.layers.Lambda.from_config = classmethod(_safe)

def fold_id(p):
    m = re.search(r"fold[_-]?(\d+)", os.path.basename(p), re.IGNORECASE)
    return int(m.group(1)) if m else None

def pair_files(models, scalers):
    m = {fold_id(p): p for p in models}
    s = {fold_id(p): p for p in scalers}
    if None in m or None in s or len(m) != len(models) or len(s) != len(scalers):
        print("[worker] WARNING: fold numbers unreadable, using sorted pairing", flush=True)
        return list(zip(sorted(models), sorted(scalers)))
    missing = (set(m) | set(s)) - (set(m) & set(s))
    if missing:
        sys.exit(f"Folds without a matching pair: {sorted(missing)}")
    return [(m[i], s[i]) for i in sorted(set(m) & set(s))]

# --- Load once for the whole batch ---
log("loading ESM-2...")
device = torch.device("cpu")
tokenizer = AutoTokenizer.from_pretrained(PLM_MODEL_NAME)
plm = EsmModel.from_pretrained(PLM_MODEL_NAME).to(device).eval()

log("loading ensemble...")
model_files = glob.glob(os.path.join(MODEL_DIR, "*.keras")) or \
              glob.glob(os.path.join(MODEL_DIR, "*.h5"))
scaler_files = glob.glob(os.path.join(MODEL_DIR, "*.joblib"))
if not model_files or not scaler_files:
    sys.exit(f"{len(model_files)} models / {len(scaler_files)} scalers in {MODEL_DIR}")

models, scalers = [], []
for mp, sp in pair_files(model_files, scaler_files):
    log(f"{os.path.basename(mp)} <-> {os.path.basename(sp)}")
    # safe_mode stays at its default (True): if a Lambda slips past the patch,
    # Keras raises a readable ValueError instead of segfaulting.
    models.append(keras.models.load_model(mp, compile=False))
    scalers.append(joblib.load(sp))
if len(set(_BYPASSED)) > 1:
    sys.exit(f"Folds disagree on length normalisation: {sorted(set(_BYPASSED))}")
log(f"{len(models)} fold(s) loaded")

# --- Per-structure feature extraction. No bare excepts: a silent fallback
# --- feature vector produces a confident-looking probability from garbage.
parser = PDBParser(QUIET=True)

def analyse(pdb_path, json_path):
    structure = parser.get_structure("s", pdb_path)[0]
    chain = next(structure.get_chains())
    sequence = "".join(protein_letters_3to1.get(r.get_resname().upper(), "X")
                       for r in chain.get_residues() if is_aa(r, standard=True))
    if not sequence:
        raise ValueError(f"No standard residues parsed from {pdb_path}")

    atoms = list(structure.get_atoms())
    n_res = len(list(structure.get_residues()))
    if not atoms or n_res == 0:
        raise ValueError(f"No atoms/residues parsed from {pdb_path}")
    com = sum(a.coord for a in atoms) / len(atoms)
    rog = float(np.sqrt(sum(np.sum((a.coord - com) ** 2) for a in atoms) / len(atoms))
                / np.sqrt(n_res))

    seq_clean = "".join(c for c in sequence if c in "ACDEFGHIKLMNPQRSTVWY")
    pa = ProteinAnalysis(seq_clean)
    biochem = [pa.isoelectric_point(), pa.gravy(), pa.aromaticity(),
               pa.molecular_weight()]

    with open(json_path) as f:
        data = json.load(f)
    plddt = np.array(data["plddt"], dtype="float32")
    pae = np.array(data["pae"], dtype="float32")

    L = min(len(sequence), len(plddt), pae.shape[0])
    effective_len = L - 5
    if effective_len <= 0:
        raise ValueError(f"Protein too short (len={L}) for -5 truncation.")
    n = min(effective_len, MAX_LENGTH)

    pad_pae = np.zeros((MAX_LENGTH, MAX_LENGTH), dtype="float32")
    pad_pae[:n, :n] = pae[:n, :n]
    pad_plddt = np.zeros(MAX_LENGTH, dtype="float32"); pad_plddt[:n] = plddt[:n]
    pad_row = np.zeros(MAX_LENGTH, dtype="float32")
    pad_col = np.zeros(MAX_LENGTH, dtype="float32")
    pad_row[:n] = np.mean(pae[:n, :n], axis=1)
    pad_col[:n] = np.mean(pae[:n, :n], axis=0)

    with torch.no_grad():
        tok = tokenizer(sequence, return_tensors="pt", truncation=True, max_length=1022)
        emb = plm(**tok).last_hidden_state.squeeze(0).mean(dim=0).cpu().numpy()

    raw_scalars = np.array(biochem + [rog], dtype="float32").reshape(1, -1)
    base = {
        "pae_input":       np.expand_dims(pad_pae, [0, -1]),
        "plddt_input":     np.expand_dims(pad_plddt, [0, -1]),
        "embedding_input": np.expand_dims(emb, 0).astype("float32"),
        "pae_row_input":   np.expand_dims(pad_row, [0, -1]),
        "pae_col_input":   np.expand_dims(pad_col, [0, -1]),
        "length_input":    np.array([[effective_len]], dtype="float32"),  # (1,1)
    }

    fold_preds = []
    for mdl, sc in zip(models, scalers):
        inp = dict(base)
        inp["scalar_features_input"] = sc.transform(raw_scalars).astype("float32")
        fold_preds.append(float(mdl.predict(inp, verbose=0)[0][0]))

    avg = float(np.mean(fold_preds))
    return {"sequence": sequence, "probability": avg,
            "label": "AMYLOID" if avg > THRESHOLD else "NON-AMYLOID",
            "threshold": THRESHOLD,
            "fold_scores": fold_preds, "rog": rog,
            "effective_len": int(effective_len), "mean_plddt": float(np.mean(plddt[:L]))}

with open(MANIFEST) as f:
    jobs = json.load(f)

out = []
for job in jobs:
    log(f"analysing {job['id']}...")
    try:
        r = analyse(job["pdb"], job["json"])
        r["id"] = job["id"]
        r["ok"] = True
    except Exception as exc:
        # One bad structure must not kill the batch.
        r = {"id": job["id"], "ok": False, "error": f"{type(exc).__name__}: {exc}"}
        print(f"[worker] ERROR {job['id']}: {r['error']}", flush=True)
    out.append(r)

with open(OUT_PATH, "w") as f:
    json.dump(out, f)
log("done")
'''

with open(SCRIPT_PATH, "w") as f:
    f.write(WORKER)
with open(MANIFEST_PATH, "w") as f:
    json.dump(manifest, f)

# ------------------------------------------------------------------------------
# 4. RUN ANALYSIS (isolated, CPU-only)
# ------------------------------------------------------------------------------
print(f"\n{'='*60}\nAnalysing {len(manifest)} structure(s)...")

env = dict(os.environ)
env["CUDA_VISIBLE_DEVICES"] = "-1"
env.pop("TF_USE_LEGACY_KERAS", None)     # models are Keras 3 format
env["TF_CPP_MIN_LOG_LEVEL"] = "3"
env["TRANSFORMERS_VERBOSITY"] = "error"
env["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
env["TOKENIZERS_PARALLELISM"] = "false"

proc = subprocess.run(
    [sys.executable, SCRIPT_PATH, os.path.join(REPO_PATH, MODEL_FOLDER_NAME),
     MANIFEST_PATH, RESULTS_PATH, "1" if VERBOSE else "0"],
    env=env, capture_output=True, text=True,
)

for line in proc.stdout.splitlines():
    if VERBOSE or "WARNING" in line or "ERROR" in line:
        print(line)

if proc.returncode != 0:
    print(proc.stderr[-4000:])
    if proc.returncode == -11:
        raise RuntimeError(
            "Worker segfaulted (SIGSEGV). Set VERBOSE = True and re-run to see "
            "how far it got. If it dies on the first load_model, TensorFlow is "
            "damaged: !pip install -q --force-reinstall --no-deps tensorflow==2.20.0"
        )
    if proc.returncode < 0:
        raise RuntimeError(f"Worker killed by signal {-proc.returncode}.")
    raise RuntimeError(f"Worker exited with code {proc.returncode}")

with open(RESULTS_PATH) as f:
    analysed = json.load(f)

# ------------------------------------------------------------------------------
# 5. COLLECT
# ------------------------------------------------------------------------------
by_id = {j["id"]: j for j in manifest}
for r in analysed:
    src = by_id[r["id"]]
    if not r.get("ok"):
        print(f"❗️ {r['id']}: {r['error']}")
        all_results.append({"ID": r["id"], "Sequence": src["input_sequence"],
                            "Prediction": "Error", "Probability": 0.0,
                            "Notes": r["error"]})
        continue

    folds = r["fold_scores"]
    spread = max(folds) - min(folds)
    note = "Success"
    if spread > 0.25:
        note = f"Success (fold spread {spread:.2f})"
    if r["mean_plddt"] < 70:
        note += f"; low mean pLDDT {r['mean_plddt']:.1f}"

    print(f"   {r['id']}: {r['label']} ({r['probability']:.4f})"
          + (f"  [folds {min(folds):.2f}–{max(folds):.2f}]" if spread > 0.25 else ""))

    all_results.append({
        "ID": r["id"], "Sequence": r["sequence"],
        "Prediction": r["label"], "Probability": r["probability"],
        "Notes": note,
        "Threshold": r["threshold"],
        "Fold_Scores": ";".join(f"{p:.6f}" for p in folds),
        "Fold_Min": min(folds), "Fold_Max": max(folds),
        "Mean_pLDDT": r["mean_plddt"], "RoG": r["rog"],
        "Effective_Len": r["effective_len"],
    })

print("\n\n✅ Batch processing complete.")

In [ ]:
#@title Download Results as CSV
import pandas as pd
from google.colab import files

# Check if the 'all_results' list exists and has content
if 'all_results' in locals() and all_results:
    # Convert the list of dictionaries to a pandas DataFrame
    results_df = pd.DataFrame(all_results)

    # Create a cleaner display column for confidence
    results_df['Confidence %'] = (results_df['Probability'] * 100).round(2)

    # Reorder columns slightly for readability
    cols = ['ID', 'Prediction', 'Probability', 'Confidence %', 'Sequence', 'Notes']
    results_df = results_df[cols]

    # Define the CSV filename
    csv_filename = 'alyzer3d_batch_results.csv'

    # Save the DataFrame to a CSV file
    results_df.to_csv(csv_filename, index=False)

    print(f"✅ Results have been saved to '{csv_filename}'.")
    print("Preview of results:")
    display(results_df.head())

    print("\nStarting download...")
    # Trigger the file download in the browser
    files.download(csv_filename)
else:
    print("❗️ No results found to download. Please run the 'Batch Prediction and Analysis' cell (Cell 2) first.")